In [1]:
import numpy as np
import pandas as pd
from pylab import *
import seaborn as sns
import pickle
import matplotlib.pyplot as plt
import pandas as pd
from pylab import *
from sequana import FastA
import tensorflow as tf
from tensorflow.keras import layers, models
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import random
from collections import defaultdict
from sklearn.metrics import classification_report


2025-07-30 11:32:28.052718: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-30 11:32:28.056787: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-30 11:32:28.068348: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753867948.088001 3826055 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753867948.093758 3826055 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753867948.109524 3826055 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
centromeres = pd.read_csv("../../output/estimation/major/Major.csv")
centromeres = {
    str(row['Chromosome']): (row['start'], row['end'])
    for _, row in centromeres.iterrows()
}


In [3]:
f = FastA("../../data/Fasta/TriTrypDB-68_LmajorFriedlin_Genome.fasta")

In [4]:
def one_hot_encoding(x):
    if x == 'A':
        return np.array([1,0,0,0])
    elif x == 'C':
        return np.array([0,1,0,0])
    elif x == 'G':
        return np.array([0,0,1,0])
    elif x == 'T':
        return np.array([0,0,0,1])
    else:
        return np.array([0,0,0,0])
        

In [5]:
SIZE=3000

In [6]:
def get_true_positive(size=3000):
    data = []
    for chrom in range(1,36+1):
        start, stop = centromeres[str(chrom)]
        #seq = f.sequences[f.names.index(str(chrom))]
        seq = f.sequences[chrom-1]

        if stop-start != 3000:
            stop = start + size
        # flip to get more positives
        data.append([one_hot_encoding(x) for x in seq[start:stop]])
        data.append([one_hot_encoding(x) for x in seq[start:stop][::-1]])
        

    return data
positives = get_true_positive(SIZE)

In [7]:
lengths = list(f.get_lengths_as_dict().values())

In [8]:

##################################### WARNING #############################################
############################### REMOVE FALSE NEGATIVE (centromeres) #######################

def get_true_negatives(N=1000,size=3000,seed=42):
    data = []
    positions = defaultdict(list)
    random.seed(seed)

    # get the random combos first
    for i in tqdm(range(N)):
        chrom = random.randint(1,36)
        N = lengths[chrom-1]
        pos = random.randint(1, N-size)
        start, stop = centromeres[str(chrom)]
        if pos>start and pos<stop:
            pass # this is a centromeres so not a negative
        else:
            positions[chrom].append(pos)
        
        
    for chrom in tqdm(positions.keys()):
        seq = f.sequences[chrom-1]
        for position in positions[chrom]:
            data.append([one_hot_encoding(x) for x in seq[position:position+size]])
    return data

    #negatives = get_true_negatives(N=10000, size=SIZE)

In [9]:
layers_values = [32]
for layer in layers_values:
    results = []
    f = FastA("../../data/Fasta/TriTrypDB-68_LmajorFriedlin_Genome.fasta")
    for i in range(1,11):
        print("###############################################################################")
        print(f'{i}/10')
        print("###############################################################################")

        negatives = get_true_negatives(N=10000, size=SIZE,seed=i)
     
        
        X_pos = positives
        X_neg = negatives
        
        
        # Create label arrays
        y_pos = [1] * len(X_pos)
        y_neg = [0] * len(X_neg)
        
        # Combine and shuffle
        X = np.array(X_pos + X_neg)  # shape: (N, 3000, 4)
        y = np.array(y_pos + y_neg)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )
    
    
        layer_size=layer
        print(layer_size)
        print(SIZE)
        model = models.Sequential([
            layers.Input(shape=(SIZE, 4)),
            layers.Conv1D(layer_size, kernel_size=15, activation='relu'),
            layers.MaxPooling1D(pool_size=2),
            layers.Conv1D(layer_size*2, kernel_size=10, activation='relu'),
            layers.GlobalMaxPooling1D(),
            layers.Dense(layer_size, activation='relu'),
            layers.Dense(1, activation='sigmoid')  # binary classification
        ])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        
        seed = 42
    
        tf.random.set_seed(seed)
        
        tf.config.experimental.enable_op_determinism()
        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=40,
            batch_size=32,
            class_weight={0: 1, 1: len(y_neg)/len(y_pos)},  # handle imbalance
            callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
        )
    
    
        y_pred = model.predict(X_test) > 0.2
        report = classification_report(y_test, y_pred, output_dict=True)
    
        metrics = {
            'f1_score': report['1']['f1-score'],
            'precision': report['1']['precision'],
            'recall': report['1']['recall']
        }
        results.append(metrics)

    with open(f"{layer}_layer_result_patience5.pkl", "wb") as f:
        pickle.dump(results, f)




###############################################################################
1/10
###############################################################################


100%|███████████████████████████████████████████| 36/36 [00:31<00:00,  1.14it/s]


32
3000


2025-07-30 11:33:46.703156: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/40


...

250/251 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 0.7783 - loss: 1.3903

2025-07-30 12:55:20.177966: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


251/251 ━━━━━━━━━━━━━━━━━━━━ 31s 117ms/step - accuracy: 0.7767 - loss: 1.3903 - val_accuracy: 0.6126 - val_loss: 0.6623
Epoch 2/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 39s 110ms/step - accuracy: 0.8689 - loss: 0.9405 - val_accuracy: 0.9591 - val_loss: 0.2298
Epoch 3/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 26s 104ms/step - accuracy: 0.9430 - loss: 0.4183 - val_accuracy: 0.9775 - val_loss: 0.1024
Epoch 4/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 42s 107ms/step - accuracy: 0.9709 - loss: 0.1640 - val_accuracy: 0.9875 - val_loss: 0.0421
Epoch 5/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 26s 105ms/step - accuracy: 0.9775 - loss: 0.1139 - val_accuracy: 0.9845 - val_loss: 0.0495
Epoch 6/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 26s 103ms/step - accuracy: 0.9798 - loss: 0.0861 - val_accuracy: 0.9890 - val_loss: 0.0321
Epoch 7/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 26s 103ms/step - accuracy: 0.9905 - loss: 0.0402 - val_accuracy: 0.9910 - val_loss: 0.0266
Epoch 8/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 27s 109ms/step - accuracy: 0.9953 - loss: 0.0241 - val

2025-07-30 13:04:24.548266: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step


In [11]:
with open(f"{64}_layer_result_patience5.pkl", "wb") as f:
    pickle.dump(results, f)


In [32]:
import pickle
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Liste des tailles de couches
layer_sizes = [16, 32, 64, 128]

# Charger les résultats et les mettre dans une liste
all_data = []

for layer in layer_sizes: 
    with open(f"{layer}_layer_result.pkl", "rb") as f:
        results = pickle.load(f)
        for metrics in results:
            all_data.append({
                'layer_size': layer,
                'f1_score': metrics['f1_score'],
                'precision': metrics['precision'],
                'recall': metrics['recall']
            })

# Convertir en DataFrame
df = pd.DataFrame(all_data)

# Afficher les 3 boxplots
metrics = ['f1_score', 'precision', 'recall']
for metric in metrics:
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='layer_size', y=metric, data=df)
    plt.title(f"Boxplot of {metric} by Layer Size")
    plt.xlabel("Layer Size")
    plt.ylabel(metric.capitalize())
    plt.grid(True)
    plt.tight_layout()
    plt.show()


(8011, 2003)